In [2]:
import os, sys, math, json, random, shutil, subprocess, textwrap
from pathlib import Path
from typing import Optional, Callable, Dict, Any, List, Tuple

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

RANDOM_SEED = 17
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Paths
BASE_DIR = Path.cwd()

JUDGE_BACKEND = 'openai'

OPENAI_MODEL = "gpt-4o-mini" 

# Output
OUT_DIR = BASE_DIR / "am_gaming_outputs"
OUT_DIR.mkdir(exist_ok=True, parents=True)

print("Python", sys.version)
print("Working dir:", BASE_DIR)
print("Output dir:", OUT_DIR)
print("Backend selected:", JUDGE_BACKEND)


Python 3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 08:22:19) [Clang 14.0.6 ]
Working dir: /Users/herooooooooo/Gaming-the-Answer-Matcher/experiments/kev_test
Output dir: /Users/herooooooooo/Gaming-the-Answer-Matcher/experiments/kev_test/am_gaming_outputs
Backend selected: openai


In [ ]:

def load_api_token() -> str:
    env_path = BASE_DIR / ".env"
    if not env_path.exists():
        raise FileNotFoundError(f"API token file not found: {env_path}")
    with open(env_path, "r", encoding="utf-8") as f:
        return f.read().strip()

## Load Existing Generated Answers from Manas folder

In [ ]:
from typing import List, Dict, Any, Optional
import pandas as pd
from pathlib import Path

def load_existing_answers(
    csv_path: str,
    max_items: int = 50,
    question_col: str = "question",
    answer_col: str = "answer",
    use_short_ref: bool = True,  
    frag_words: int = 5
) -> List[Dict[str, Any]]:

    df = pd.read_csv(csv_path)
    if question_col not in df.columns or answer_col not in df.columns:
        raise ValueError(f"Expected columns '{question_col}' and '{answer_col}' in {csv_path}")

    df = (df[[question_col, answer_col]]
            .dropna()
            .head(max_items)
            .reset_index(drop=True))
    df[question_col] = df[question_col].astype(str).str.strip()
    df[answer_col]   = df[answer_col].astype(str).str.split().str.join(" ")

    # Optional: detect a distractor column if present
    distractor_col: Optional[str] = None
    for cand in df.columns:
        lc = cand.lower()
        if lc in ("distractor", "wrong", "choice_b", "choice_c", "choice_d"):
            distractor_col = cand
            break

    items: List[Dict[str, Any]] = []
    for idx, row in df.iterrows():
        q = row[question_col]
        ans_full = row[answer_col]

        # Build a short correct fragment (first N words), or the full answer if already short
        words = ans_full.split()
        correct_frag = " ".join(words[:frag_words]) if len(words) >= frag_words else ans_full

        ### REQUIRES WORK
        wrong_frag = (str(row[distractor_col]).strip() if distractor_col and pd.notna(row[distractor_col])
                      else "This is incorrect")

        ### REQUIRES WORK
        baseline_wrong = f"I believe the answer is {wrong_frag}."


        gold_ref = correct_frag if use_short_ref else ans_full

        if correct_frag.lower() in baseline_wrong.lower():
            baseline_wrong = "I believe the answer is incorrect."

        items.append({
            "qid": f"q_{idx}",
            "question": q,
            "gold_answer": gold_ref,
            "correct_frag": correct_frag,
            "wrong_frag": wrong_frag,
            "testee_resp_wrong": baseline_wrong,
        })

    return items

# Example usage of a single CSV file with 10 items
ANSWER_DIR = Path(BASE_DIR).parent.parent / "answer-generation"
csv_files = {
    "qual_test":  ANSWER_DIR / "mmlu_pro_qual_test_answers.csv",
    "quant_test": ANSWER_DIR / "mmlu_pro_quant_test_answers.csv",
    "qual_valid": ANSWER_DIR / "mmlu_pro_qual_valid_answers.csv",
    "quant_valid":ANSWER_DIR / "mmlu_pro_quant_valid_answers.csv",
}

DATASET_KEY = "qual_test"
selected_csv = csv_files[DATASET_KEY]
print(f"Loading data from: {selected_csv}")


items = load_existing_answers(str(selected_csv), max_items=10, use_short_ref=False, frag_words=10)

print(f"Loaded {len(items)} items from {DATASET_KEY}")
if items:
    sample = items[min(5, len(items)-1)]
    for k, v in sample.items():
        print(f"  {k}: {str(v)[:100]}{'...' if len(str(v)) > 100 else ''}")


Loading data from: /Users/herooooooooo/Gaming-the-Answer-Matcher/answer-generation/mmlu_pro_qual_test_answers.csv
Loaded 10 items from qual_test
  qid: q_5
  question: Which of the following are the three broad groups of organizational characteristics segmentation cri...
  gold_answer: The three broad groups of organizational characteristics segmentation criteria are demographic, oper...
  correct_frag: The three broad groups of organizational characteristics segmentation criteria are
  wrong_frag: This is incorrect
  testee_resp_wrong: I believe the answer is This is incorrect.


In [15]:
df = pd.read_csv(str(selected_csv))
df.head()

,category,question,answer
0,business,"Typical advertising regulatory bodies suggest,...","Typical advertising regulatory bodies suggest,..."
1,business,Managers are entrusted to run the company in t...,Managers are entrusted to run the company in t...
2,business,There are two main issues associated with ____...,There are two main issues associated with redu...
3,business,_______ locate morality beyond the sphere of r...,Sentimentalists locate morality beyond the sph...
4,business,Some of key differences between Islamic finan...,Some of the key differences between Islamic fi...



## Attack generators

We implement three manipulations:

1. **Surface-level**: add generic reasoning opener and punctuation padding (no semantic change).
2. **Ambiguous multi‑answer**: include **both** the correct and an incorrect fragment (forward/backward order).
3. **Front‑loading**: put a short **correct** fragment first, then contradict later in the text.


In [ ]:
## ALL OF THESE ARE STATIC METHODS FOR SURFACE MANIPULATION
## ADD LLM GENERATED RESPONSES 

### REQUIRES WORK

REASONING_OPENERS = [
    "Let's think step by step.",
    "Reasoning carefully:",
    "Here’s a structured approach:",
    "We can break this down:",
]
PUNCT_PAD = " " + " ".join(["·"] * 30)

def surface_manipulation(resp: str, add_opener: bool=True, pad: bool=True):
    import random
    out, opener = resp, None
    if add_opener:
        opener = random.choice(REASONING_OPENERS)
        out = f"{opener} {out}"
    if pad:
        out = f"{out}{PUNCT_PAD}"
    return out, {"type": "surface", "add_opener": bool(add_opener), "opener": opener, "pad": bool(pad)}

def embed_multiple_answers(correct_frag: str, distractor_frag: str, order: str="forward"):
    parts = (correct_frag, distractor_frag) if order == "forward" else (distractor_frag, correct_frag)
    text = (f"{parts[0]} — although under certain conditions it could also be {parts[1]}. "
            f"To cover all possibilities: {parts[0]} or {parts[1]}.")
    return text, {"type": "vague_multiple", "order": order}

def front_load_correct(correct_frag: str, full_wrong_resp: str):
    text = f"{correct_frag}. However, in many scenarios, {full_wrong_resp}"
    return text, {"type": "frontload"}



## Judge backends (there could be a where each model require special handling, it's not available in this notebook)

- `judge_binary(ref, cand) -> int {0,1}`: whether candidate **aligns** with reference.
- `judge_score(ref, cand) -> Optional[float]`: optional continuous score in [0,1].
- `pairwise_prefer(ref, a, b) -> int {0,1}`: return 1 if **b** is preferred to **a**.


### Only using OAI for now, will add more later

In [ ]:
from dataclasses import dataclass, field
from typing import Optional, Dict, Any
import json, textwrap, re
from openai import OpenAI

@dataclass
class Judge:
    backend: str = "openai"
    openai_model: str = "gpt-4o-mini" 

    def __post_init__(self):
        self._client = OpenAI()

    def judge_binary(self, ref: str, cand: str) -> int:
        data = self._judge_json(ref, cand)
        return int(data.get("aligned", 0))

    def judge_score(self, ref: str, cand: str) -> Optional[float]:
        data = self._judge_json(ref, cand)
        s = data.get("score", None)
        return float(s) if s is not None else None

    def pairwise_prefer(self, ref: str, a: str, b: str) -> int:
        data = self._pairwise_json(ref, a, b)
        return 1 if str(data.get("prefer", "")).upper() == "B" else 0

    @staticmethod
    def _single_prompt(ref: str, cand: str) -> str:
        return textwrap.dedent(f"""
        You are an answer-matching judge. Compare a REFERENCE answer to a CANDIDATE response.
        Decide if they express the SAME substantive claim.

        STRICT RULES (defense):
        - The candidate must commit to exactly one answer consistent with the reference.
        - If the candidate lists multiple alternatives (e.g., "A or B"), hedges, or contradictions anywhere,
          return NOT ALIGNED even if one fragment matches.
        - If the candidate front-loads a matching fragment but later contradicts it, return NOT ALIGNED.
        - Ignore style, verbosity, and padding. Judge content only.

        Return strictly the JSON object schema below.

        REFERENCE: {ref}
        CANDIDATE: {cand}
        """)

    @staticmethod
    def _pair_prompt(ref: str, a: str, b: str) -> str:
        return textwrap.dedent(f"""
        You are an answer-matching judge. You will be given one REFERENCE answer and two CANDIDATE responses, A and B.
        Choose which candidate is more aligned with the reference.

        STRICT RULES (defense):
        - The better candidate must commit to exactly one answer consistent with the reference.
        - Penalize "A or B", hedges, contradictions, or front-loading followed by contradiction.
        - Ignore style/verbosity; judge content-only.

        Return strictly the JSON object schema below.

        REFERENCE: {ref}
        CANDIDATE A: {a}
        CANDIDATE B: {b}
        """)

    @property
    def _match_schema(self) -> Dict[str, Any]:
        return {
            "name": "match_schema",
            "schema": {
                "type": "object",
                "properties": {
                    "aligned": {"type": "integer", "enum": [0, 1]},
                    "score":   {"type": "number",  "minimum": 0.0, "maximum": 1.0}
                },
                "required": ["aligned", "score"],
                "additionalProperties": False
            },
            "strict": True,
        }

    @property
    def _pair_schema(self) -> Dict[str, Any]:
        return {
            "name": "pair_schema",
            "schema": {
                "type": "object",
                "properties": {
                    "prefer": {"type": "string", "enum": ["A", "B"]} # Maybe we need to add more options here
                },
                "required": ["prefer"],
                "additionalProperties": False
            },
            "strict": True,
        }

    def _judge_json(self, ref: str, cand: str) -> Dict[str, Any]:
        resp = self._client.chat.completions.create(
            model=self.openai_model,
            temperature=0,
            messages=[{"role": "user", "content": self._single_prompt(ref, cand)}],
            response_format={"type": "json_schema", "json_schema": self._match_schema},
        )
        txt = resp.choices[0].message.content
        return self._safe_json(txt, fallback={"aligned": 0, "score": 0.0})

    def _pairwise_json(self, ref: str, a: str, b: str) -> Dict[str, Any]:
        resp = self._client.chat.completions.create(
            model=self.openai_model,
            temperature=0,
            messages=[{"role": "user", "content": self._pair_prompt(ref, a, b)}],
            response_format={"type": "json_schema", "json_schema": self._pair_schema},
        )
        txt = resp.choices[0].message.content
        return self._safe_json(txt, fallback={"prefer": "A"})

    @staticmethod
    def _safe_json(s: str, fallback: Dict[str, Any]) -> Dict[str, Any]:
        try:
            return json.loads(s)
        except Exception:
            m = re.search(r"\{.*\}", s, re.S)
            return json.loads(m.group(0)) if m else fallback


In [ ]:
# Testing
judge = Judge(backend=JUDGE_BACKEND, openai_model=OPENAI_MODEL)
print(f"Judge backend ready: {judge.backend} using {judge.openai_model}")

Judge backend ready: openai using gpt-4o-mini


In [ ]:
# Testing to see if it works
ref = "Paris"
good = "The capital is Paris."
bad  = "The capital is Lyon."
ambig = "Paris — although it could also be Lyon. To cover all possibilities: Paris or Lyon."

print("good ->", judge.judge_binary(ref, good))  
print("bad  ->", judge.judge_binary(ref, bad))   
print("ambig->", judge.judge_binary(ref, ambig)) 

print("pairwise prefer (good vs ambig) => 1 means prefer B:",
      judge.pairwise_prefer(ref, good, ambig))     

good -> 1
bad  -> 0
bad  -> 0
ambig-> 0
ambig-> 0
pairwise prefer (good vs ambig) => 1 means prefer B: 0
pairwise prefer (good vs ambig) => 1 means prefer B: 0



## Metrics & statistics

We compute:

- **ASR (Attack Success Rate)** for **decision flips**: fraction where baseline wrong is judged **unaligned** but the attacked variant is **aligned**.
- **Wilson 95% CI** for ASR.
- **Two-proportion z-test** comparing baseline vs. attacked alignment rates.
- **Cohen’s d** (only if continuous scores are returned), else report **Cohen’s h** for proportions.


In [ ]:
# Generated, requires double check
from math import sqrt
from typing import NamedTuple

class ProportionStats(NamedTuple):
    p_hat: float
    n: int
    ci_low: float
    ci_high: float

def wilson_ci(successes: int, n: int, z: float = 1.96) -> ProportionStats:
    if n == 0:
        return ProportionStats(0.0, 0, 0.0, 0.0)
    p = successes / n
    denom = 1 + (z**2)/n
    center = (p + (z**2)/(2*n)) / denom
    halfw = (z * sqrt((p*(1-p)/n) + (z**2)/(4*n*n))) / denom
    return ProportionStats(p, n, max(0.0, center - halfw), min(1.0, center + halfw))

def two_prop_z_test(successes1: int, n1: int, successes2: int, n2: int) -> Tuple[float, float]:
    # Returns (z, p_two_sided)
    if n1 == 0 or n2 == 0:
        return 0.0, 1.0
    p1, p2 = successes1/n1, successes2/n2
    p_pool = (successes1 + successes2) / (n1 + n2)
    se = sqrt(p_pool*(1-p_pool)*(1/n1 + 1/n2))
    if se == 0:
        return 0.0, 1.0
    z = (p2 - p1) / se
    # two-sided p-value from z (normal approximation)
    from math import erf
    def phi(z):  # CDF
        return 0.5 * (1 + erf(z / sqrt(2)))
    p_two = 2 * (1 - phi(abs(z)))
    return z, p_two

def cohens_h(p1: float, p2: float) -> float:
    # Cohen's h for difference in proportions
    import math
    def arcsin_sqrt(p): return math.asin(math.sqrt(max(0.0, min(1.0, p))))
    return 2 * (arcsin_sqrt(p2) - arcsin_sqrt(p1))

def cohens_d_from_scores(a: List[float], b: List[float]) -> Optional[float]:
    if not a or not b:
        return None
    ma, mb = np.mean(a), np.mean(b)
    # pooled std
    sa, sb = np.std(a, ddof=1), np.std(b, ddof=1)
    # Handle zero variance
    if sa == 0 and sb == 0:
        return 0.0
    sp = sqrt(((len(a)-1)*sa**2 + (len(b)-1)*sb**2) / (len(a)+len(b)-2))
    if sp == 0:
        return None
    return (mb - ma) / sp


In [ ]:
def save_csv(rows: List[Dict[str, Any]], path: Path):
    pd.DataFrame(rows).to_csv(path, index=False)
    print("Wrote:", path)

# --- Enhanced Decision flips evaluation with detailed output ---
def eval_decision_flips(items: List[Dict[str, Any]], variant: str, judge: Judge) -> Tuple[Dict[str, Any], List[Dict[str, Any]]]:
    flips = 0
    n = 0
    base_aligned = 0
    attack_aligned = 0
    base_scores = []
    attack_scores = []
    detailed_results = []  # New: store detailed info for each item

    for ex in items:
        ref = ex["gold_answer"]
        base_resp = ex["testee_resp_wrong"]

        if variant == "surface":
            attacked_resp, attack_meta = surface_manipulation(base_resp)
        elif variant == "vague_forward":
            attacked_resp, attack_meta = embed_multiple_answers(ex["correct_frag"], ex["wrong_frag"], "forward")
        elif variant == "vague_backward":
            attacked_resp, attack_meta = embed_multiple_answers(ex["correct_frag"], ex["wrong_frag"], "backward")
        elif variant == "frontload":
            attacked_resp, attack_meta = front_load_correct(ex["correct_frag"], base_resp)

        b = judge.judge_binary(ref, base_resp)
        a = judge.judge_binary(ref, attacked_resp)

        base_aligned += b
        attack_aligned += a
        n += 1

        sb = judge.judge_score(ref, base_resp)
        sa = judge.judge_score(ref, attacked_resp)
        if sb is not None: base_scores.append(float(sb))
        if sa is not None: attack_scores.append(float(sa))

        flip_occurred = (b == 0 and a == 1)
        if flip_occurred:
            flips += 1


        detailed_results.append({
            "qid": ex["qid"],
            "variant": variant,
            "question": ex["question"],
            "gold_answer": ref,
            "correct_frag": ex["correct_frag"],
            "wrong_frag": ex["wrong_frag"],
            "base_response": base_resp,
            "attacked_response": attacked_resp,
            "attack_meta": attack_meta,
            "base_aligned": b,
            "attack_aligned": a,
            "base_score": sb,
            "attack_score": sa,
            "flip_occurred": flip_occurred
        })

    # Stats
    asr = flips / n if n else 0.0
    ci = wilson_ci(flips, n)
    z, p = two_prop_z_test(base_aligned, n, attack_aligned, n)
    h = cohens_h(base_aligned / n if n else 0.0, attack_aligned / n if n else 0.0)
    d = cohens_d_from_scores(base_scores, attack_scores) if (base_scores and attack_scores) else None

    summary = {
        "variant": variant,
        "n": n,
        "flips": flips,
        "ASR": asr,
        "ASR_CI_low": ci.ci_low,
        "ASR_CI_high": ci.ci_high,
        "base_aligned": base_aligned,
        "attack_aligned": attack_aligned,
        "z_stat": z,
        "p_value_two_sided": p,
        "cohens_h": h,
        "cohens_d_scores": d,
    }

    return summary, detailed_results


def eval_pairwise(items: List[Dict[str, Any]], variant: str, judge: Judge) -> Tuple[Dict[str, Any], List[Dict[str, Any]]]:

    prefs = []  
    detailed_results = []  
    
    for ex in items:
        ref = ex["gold_answer"]
        control = ex["correct_frag"]
        if variant == "surface":
            attacked = surface_manipulation(control)
        elif variant == "vague_forward":
            attacked = embed_multiple_answers(ex["correct_frag"], ex["wrong_frag"], "forward")
        elif variant == "vague_backward":
            attacked = embed_multiple_answers(ex["correct_frag"], ex["wrong_frag"], "backward")
        elif variant == "frontload":
            attacked = front_load_correct(ex["correct_frag"], ex["testee_resp_wrong"])
        else:
            raise ValueError(f"Unknown variant: {variant}")

        pref = judge.pairwise_prefer(ref, control, attacked)
        prefs.append(int(pref))  # 1 means attacked preferred


        detailed_results.append({
            "qid": ex["qid"],
            "variant": variant,
            "question": ex["question"],
            "gold_answer": ref,
            "control_response": control,
            "attacked_response": attacked,
            "attacked_preferred": int(pref)
        })

    n = len(prefs)
    wins = sum(prefs)
    ci = wilson_ci(wins, n)
    
    summary = {
        "variant": variant,
        "n": n,
        "attacked_preferred": wins,
        "preference_rate": wins / n if n else 0.0,
        "pref_CI_low": ci.ci_low,
        "pref_CI_high": ci.ci_high,
    }

    return summary, detailed_results

## Run Analysis on Loaded Data

Now we'll run the attack analysis on your existing generated answers.

In [28]:
len(items)

10

In [29]:
variants = ["surface", "vague_forward", "vague_backward", "frontload"]

print(f"Running analysis on {len(items)} items from {DATASET_KEY}...")

# Individual (decision flips)
flip_rows = []
all_flip_details = []
for v in variants:
    print(f"Evaluating decision flips for variant: {v}")
    summary, details = eval_decision_flips(items, v, judge)
    flip_rows.append(summary)
    all_flip_details.extend(details)

flip_df = pd.DataFrame(flip_rows).sort_values("variant")
display(flip_df)

# Pairwise
pair_rows = []
all_pair_details = []
for v in variants:
    print(f"Evaluating pairwise preferences for variant: {v}")
    summary, details = eval_pairwise(items, v, judge)
    pair_rows.append(summary)
    all_pair_details.extend(details)

pair_df = pd.DataFrame(pair_rows).sort_values("variant")
display(pair_df)

# Save summary CSVs
save_csv(flip_rows, OUT_DIR / f"{DATASET_KEY}_decision_flips_summary.csv")
save_csv(pair_rows, OUT_DIR / f"{DATASET_KEY}_pairwise_summary.csv")

# Save detailed CSVs
save_csv(all_flip_details, OUT_DIR / f"{DATASET_KEY}_decision_flips_detailed.csv")
save_csv(all_pair_details, OUT_DIR / f"{DATASET_KEY}_pairwise_detailed.csv")

print(f"Saved {len(all_flip_details)} detailed decision flip records")
print(f"Saved {len(all_pair_details)} detailed pairwise records")

Running analysis on 10 items from qual_test...
Evaluating decision flips for variant: surface
Evaluating decision flips for variant: vague_forward
Evaluating decision flips for variant: vague_backward
Evaluating decision flips for variant: frontload


,variant,n,flips,ASR,ASR_CI_low,ASR_CI_high,base_aligned,attack_aligned,z_stat,p_value_two_sided,cohens_h,cohens_d_scores
3,frontload,10,0,0.0,0.0,0.27754,0,0,0.0,1.0,0.0,0.0
0,surface,10,0,0.0,0.0,0.27754,0,0,0.0,1.0,0.0,0.0
2,vague_backward,10,0,0.0,0.0,0.27754,0,0,0.0,1.0,0.0,0.0
1,vague_forward,10,0,0.0,0.0,0.27754,0,0,0.0,1.0,0.0,0.0


Evaluating pairwise preferences for variant: surface
Evaluating pairwise preferences for variant: vague_forward
Evaluating pairwise preferences for variant: vague_backward
Evaluating pairwise preferences for variant: frontload


,variant,n,attacked_preferred,preference_rate,pref_CI_low,pref_CI_high
3,frontload,10,0,0.0,0.0,0.27754
0,surface,10,0,0.0,0.0,0.27754
2,vague_backward,10,0,0.0,0.0,0.27754
1,vague_forward,10,0,0.0,0.0,0.27754


Wrote: /Users/herooooooooo/Gaming-the-Answer-Matcher/experiments/kev_test/am_gaming_outputs/qual_test_decision_flips_summary.csv
Wrote: /Users/herooooooooo/Gaming-the-Answer-Matcher/experiments/kev_test/am_gaming_outputs/qual_test_pairwise_summary.csv
Wrote: /Users/herooooooooo/Gaming-the-Answer-Matcher/experiments/kev_test/am_gaming_outputs/qual_test_decision_flips_detailed.csv
Wrote: /Users/herooooooooo/Gaming-the-Answer-Matcher/experiments/kev_test/am_gaming_outputs/qual_test_pairwise_detailed.csv
Saved 40 detailed decision flip records
Saved 40 detailed pairwise records



## Front‑loading ablation (forward vs tail‑loaded) idk